In [2]:
import torch

In [3]:
def generate_rotation_matrix(dim, seed, device):
    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed)
    matrix = torch.randn(dim, dim, generator=generator)
    q, r = torch.linalg.qr(matrix)
    diag_sign = torch.sign(torch.diag(r))
    diag_sign[diag_sign == 0] = 1.0
    q = q * diag_sign.unsqueeze(0)
    return q.to(device)

def rotate(tensor: torch.Tensor, rotation: torch.Tensor):
    flat = tensor.float()

    # Normalise to unit sphere and store norms
    norms = flat.norm(dim=-1, keepdim=True)
    unit = flat / (norms + 1e-8)

    # Rotate and quantise
    return unit @ rotation.T, norms

def decode(tensor, rotation, norms) -> torch.Tensor:
    # Inverse rotate
    unrotated = tensor @ rotation

    # Rescale by norms
    return unrotated * norms

In [5]:
# compare padded and non-padded versions - does padding affect quantisation quality?
chunk_size = 128
ranks = [16, 32, 64]
tol = 1e-5

As = [torch.randn(chunk_size, rank) for rank in ranks]


# now do the same but with independent rotations for each A
ind_rotations = [generate_rotation_matrix(rank, seed=0, device="cpu") for rank in ranks]
rotated_As = [
    rotate(A, ind_rotation) for A, ind_rotation in zip(As, ind_rotations)
]
unrotated_As = [
    decode(rot, ind_rotation, norms)    
    for (rot, norms), ind_rotation in zip(rotated_As, ind_rotations)
]

# now pad and concatenate the As and rotate them as well
rotation = generate_rotation_matrix(max(ranks), seed=0, device="cpu")
padded_As = [torch.nn.functional.pad(A, (0, max(ranks) - A.shape[1])) for A in As]
concat_As = torch.cat(padded_As, dim=0)
rotated_concat_A, norms = rotate(concat_As, rotation)
unrotated_concat_A = decode(rotated_concat_A, rotation, norms)
unrotated_unconcat_As = []
start = 0
for rank in ranks:
    unrotated_unconcat_As.append(unrotated_concat_A[start:start+chunk_size, :rank])
    start += chunk_size

print("Comparing unrotated As with original As:")
for A, unrotated_A, unrotated_unconcat_A in zip(As, unrotated_As, unrotated_unconcat_As):
    print(torch.allclose(A, unrotated_A, atol=tol))
    print(torch.allclose(A, unrotated_unconcat_A, atol=tol))

Comparing unrotated As with original As:
True
True
True
True
True
True
